# Informações Gerais sobre o Conteúdo

---

### Uso de Inteligência Artificial
Este projeto utilizou ferramentas de IA como assistentes de coprodução. O objetivo principal deste documento é servir como uma fonte sólida de dados e material para estudos posteriores.

* **O que foi aprimorado por IA:** A otimização de trechos de código e o refinamento técnico das descrições dos métodos.
* **O que é de autoria própria:** Toda a formulação teórica, estruturação do projeto, arquitetura da API e a lógica de utilização foram planejadas e escritas pelo autor, assim como a elaboração da grande maioria dos códigos contidos aqui.

> **Resumo:** A IA funcionou como um copiloto para refinar o trabalho, mas a direção, a arquitetura e a execução principal são 100% humanas.

In [2]:
#Imports
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from numpy.linalg import svd
from scipy.stats import chi2, norm, f
from IPython.display import display, Markdown
from numba import njit
import os 

try:
    from coppeliasim_zmqremoteapi_client import RemoteAPIClient 
    print("Biblioteca CoppeliasimZMQ já instalada")
except:
    print("Biblioteca CoppeliasimZMQ NÃO instalada")
    !pip install coppeliasim-zmqremoteapi-client
    from coppeliasim_zmqremoteapi_client import RemoteAPIClient 
    print("Biblitoeca instalada com sucesso!")

Biblioteca CoppeliasimZMQ já instalada


# Sobre a atividade

Aqui será desenvolvido um algorítmo para controle de um robô num ambiente desconhecido mediante a técnica Q-Learning, e finaliznado com a técnica TOSL-iBSR (True Online SARSA Informed-Based softmax regression)

- **Primeira etapa:** Desenvolvimento dos algorítmos básicos (Controle do robô)
- **Segunda etapa:** Cadeias de Markov (Teoria e implementações)
- **Terceira etapa:** Algorítmo de recompensa e Reinforced Learning
- **Quarta etapa:** Arquitetura TOSL-iBSR
- **Quinta Etapa:** Testes de rendimento para cada arquitetura

# 1. Primeira Etapa: Controle do Robô

## 1.1 Simulação no Coppelia

### Arquitetura do Sistema de Simulação

Este ecossistema utiliza a biblioteca de API Remota baseada em ZeroMQ (`coppeliasim_zmqremoteapi_client`) para estabelecer comunicação síncrona bidirecional com o simulador CoppeliaSim. A arquitetura divide-se logicamente em três componentes principais estruturados em Programação Orientada a Objetos (POO):

* **Classe `Simulation`**: Orquestrador genérico do loop de simulação. Gerencia a inicialização da conexão de rede, o relógio interno do motor de física e força o modo de execução síncrono.
* **Classe `Pioneer`**: Interface de baixo nível que abstrai o hardware do robô, lidando com a leitura de sensores reais do cenário e a atuação nos motores.
* **Classe `LyapunovController`**: Implementação matemática do algoritmo de controle de postura em malha fechada.

In [4]:
class Simulation:
    def __init__(self, max_time=30.0):
        self.max_time = max_time
        self.current_time = 0.0
        self.is_running = False
        
        # Initialize Remote API Client connection
        self.client = RemoteAPIClient()
        self.sim = self.client.getObject('sim')
        
        self.robot = None
        self.controller = None

    def setup(self):
        """Override to configure your scene objects."""
        pass

    def loop(self):
        """Override to program execution logic."""
        pass

    def run(self):
        """Executes the synchronous simulation step loop inside CoppeliaSim."""
        self.setup()
        
        if self.robot is None:
            raise TypeError("Runtime Error: 'self.robot' must be initialized inside setup().")

        # Enable Synchronous stepping mode
        self.sim.setStepping(True)
        self.sim.startSimulation()
        self.is_running = True
        print("CoppeliaSim synchronous simulation initialized.")
        
        while self.is_running and self.current_time < self.max_time:
            # Get internal simulation clock time from engine
            self.current_time = self.sim.getSimulationTime()
            
            # Execute user-overridden logic loop
            self.loop()
            
            # Trigger physics step computation engine forward
            self.sim.step()

            #Add some timeout in the controller?
            
        # Stopping sequence execution
        self.sim.stopSimulation()
        print(f"Simulation execution halted at {self.current_time:.2f} seconds.")

## 1.2. Definindo robô Pionner

### O Robô Pioneer e a Abstração de Hardware

O robô que será utilizado nesta simulação é o **Pioneer**. Este modelo possui um anel com **16 sensores ultrassônicos** associados a ele, o que o torna uma plataforma altamente interessante para tarefas de navegação e percepção de ambiente. 

Para integrá-lo adequadamente, e considerando que estamos operando via API ZeroMQ do CoppeliaSim, será necessário definir uma classe dedicada (a classe `Pioneer`). Essa abordagem orientada a objetos nos permite abstrair toda a complexidade do controle de baixo nível, organizando a interação com o simulador em três frentes principais:

* **Percepção (Sensores):** Leitura contínua do array de ultrassons para medir a distância de possíveis obstáculos.
* **Cinemática (Atuação):** Conversão das velocidades de controle linear e angular em comandos de rotação individuais para os motores direito e esquerdo.
* **Localização (Pose):** Extração precisa da **pose** global do robô — ou seja, suas coordenadas espaciais cartesianas ($x, y$) e sua orientação/guiada ($\theta$) — em relação ao referencial absoluto do ambiente de simulação.

Essa estruturação torna o código limpo, modular e pronto para ser reutilizado facilmente em experimentos futuros.ros.

In [14]:
class Pioneer:
    def __init__(self, sim, robot_path='/PioneerP3DX'):
        """
        sim: The CoppeliaSim API instance object ('sim')
        robot_path: The base path of the Pioneer model in the scene hierarchy
        """
        self.sim = sim
        
        # Pioneer 3-DX physical specifications
        self.r = 0.0975  # Wheel radius (meters)
        self.L = 0.381   # Wheelbase track width (meters)

        # max 
        self.v_max = 1.6 # m/s
        self.w_max = 5.23 # rad/s
        
        # Fetch object handles from the CoppeliaSim scene hierarchy
        self.robot_handle = self.sim.getObject(robot_path)
        self.left_motor = self.sim.getObject(f"{robot_path}/leftMotor")
        self.right_motor = self.sim.getObject(f"{robot_path}/rightMotor")
        
        # Fetch handles for the 16 proximity sensors (sonars)
        # Note: Adjust the naming format string if your scene uses a different pattern
        self.sonar_handles = []
        for i in range(16):
            try:
                handle = self.sim.getObject(f"{robot_path}/sonar[{i}]")
                self.sonar_handles.append(handle)
            except Exception:
                # Fallback for alternative scene configurations (e.g., ultrasonicSensor[x])
                handle = self.sim.getObject(f"{robot_path}/ultrasonicSensor[{i}]")
                self.sonar_handles.append(handle)

    def get_pose(self):
        """
        Retrieves the absolute 2D pose (x, y, theta) from the simulator.
        Uses sim.handle_world (-1) as the reference frame.
        """
        position = self.sim.getObjectPosition(self.robot_handle, self.sim.handle_world)
        orientation = self.sim.getObjectOrientation(self.robot_handle, self.sim.handle_world)
        
        x = position[0]
        y = position[1]
        theta = orientation[2]  # Yaw angle around Z axis
        return np.array([x, y, theta])

    def send_commands(self, v, omega):
        """
        Inverse Kinematics: Computes wheel velocities and sends them to CoppeliaSim.
        """
        #clip velocities
        v = np.clip(v, -self.v_max, self.v_max)
        omega = np.clip(omega, -self.w_max, self.w_max)
        
        # Diferential equations
        w_r = (v + (omega * self.L) / 2.0) / self.r
        w_l = (v - (omega * self.L) / 2.0) / self.r

        
        self.sim.setJointTargetVelocity(self.left_motor, w_l)
        self.sim.setJointTargetVelocity(self.right_motor, w_r)

    def read_sonars(self):
        """
        Reads the 16 proximity sensors from the scene.
        Returns a NumPy array containing the detected distances.
        """
        distances = []
        for handle in self.sonar_handles:
            # sim.readProximitySensor returns: [result, distance, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector]
            result, distance, _, _, _ = self.sim.readProximitySensor(handle)
            
            if result > 0:
                distances.append(distance)
            else:
                distances.append(5.0)  # Max range default value if no obstacle detected
                
        return np.array(distances)

# Class to a Dummy in the simulation
class Dummy:
    def __init__(self, sim, obj_path=None):
        """
        sim: The CoppeliaSim API instance object ('sim')
        robot_path: The base path of the Pioneer model in the scene hierarchy
        """
        self.sim = sim
        
        # Fetch object handles from the CoppeliaSim scene hierarchy
        self.obj_handle = self.sim.getObject(obj_path)
        
    def get_pose(self):
        """
        Retrieves the absolute 2D pose (x, y, theta) from the simulator.
        Uses sim.handle_world (-1) as the reference frame.
        """
        position = self.sim.getObjectPosition(self.obj_handle, self.sim.handle_world)
        orientation = self.sim.getObjectOrientation(self.obj_handle, self.sim.handle_world)
        
        x = position[0]
        y = position[1]
        theta = orientation[2]  # Yaw angle around Z axis
        return np.array([x, y, theta])

## 1.3. Controlador Diferencial (Não-holonômico)

### Controlador de Postura Baseado em Lyapunov

Devido às restrições não-holonômicas de robôs de acionamento diferencial (como o Pioneer, que não pode se deslocar lateralmente), o controle direto de posição em coordenadas cartesianas não é adequado e pode gerar instabilidade. Para contornar isso, implementamos um controlador de estabilização de postura baseado no método de Lyapunov, que transforma o problema para o espaço de coordenadas polares.

O controle calcula a atuação com base em um sistema de três erros geométricos:

* **$\rho$ (Erro de Distância)**: A distância euclidiana entre a posição atual do robô e a coordenada alvo.
* **$\alpha$ (Erro de Alinhamento)**: O ângulo entre a orientação atual do robô e a linha reta que o conecta ao objetivo.
* **$\beta$ (Erro de Orientação Final)**: A diferença entre o ângulo desejado de chegada e o vetor de aproximação.

#### Leis de Controle e Estabilidade

A partir da definição de uma função candidata de Lyapunov positiva definida, derivamos as seguintes leis de controle não-lineares para calcular a velocidade linear ($v$) e angular ($\omega$) a cada iteração:

$$v = k_\rho \cdot \rho \cdot \cos(\alpha)$$

$$\omega = k_\alpha \cdot \alpha + k_\rho \cdot \frac{\sin(\alpha)\cos(\alpha)}{\alpha} \cdot (\alpha + k_\beta \cdot \beta)$$

Para garantir matematicamente a estabilidade assintótica do sistema (ou seja, que o robô sempre convergirá para o alvo sem oscilar indefinidamente), os ganhos de sintonia do controlador devem respeitar rigorosamente as seguintes condições:

* $k_\rho > 0$
* $k_\alpha - k_\rho > 0$
* $k_\beta < 0$

In [7]:
class LyapunovController:
    def __init__(self, k_rho=0.6, k_alpha=2.0, k_beta=-0.3):
        self.k_rho = k_rho
        self.k_alpha = k_alpha
        self.k_beta = k_beta

    def calculate_control(self, current_pose, target_pose):
        """
        Vectorized tracking logic based on the 3 polar error system.
        """
        x, y, theta = current_pose
        x_g, y_g, theta_g = target_pose

        dx = x_g - x
        dy = y_g - y
        
        rho = np.hypot(dx, dy)
        
        # Dead-zone convergence logic
        if rho < 0.05:
            beta = theta_g - theta
            beta = np.arctan2(np.sin(beta), np.cos(beta))
            return 0.0, self.k_alpha * beta 

        gamma = np.arctan2(dy, dx)
        alpha = np.arctan2(np.sin(gamma - theta), np.cos(gamma - theta)) 
        beta = np.arctan2(np.sin(theta_g - gamma), np.cos(theta_g - gamma)) 

        # Control equations for forward navigation tracking
        if np.abs(alpha) <= np.pi / 2.0:
            v = self.k_rho * rho * np.cos(alpha)
            omega = self.k_alpha * alpha + self.k_rho * (np.sin(alpha) * np.cos(alpha) / alpha) * (alpha + self.k_beta * beta) if alpha != 0 else self.k_alpha * alpha
        else:
            # Backward navigation tracking modifications
            alpha_b = np.arctan2(np.sin(alpha + np.pi), np.cos(alpha + np.pi))
            beta_b = np.arctan2(np.sin(beta + np.pi), np.cos(beta + np.pi))
            v = -self.k_rho * rho * np.cos(alpha_b)
            omega = self.k_alpha * alpha_b + self.k_rho * (np.sin(alpha_b) * np.cos(alpha_b) / alpha_b) * (alpha_b + self.k_beta * beta_b) if alpha_b != 0 else self.k_alpha * alpha_b

        # Hardware safety clipping limits
        return np.clip(v, -0.7, 0.7), np.clip(omega, -1.2, 1.2)

## 1.4. Exemplo de utilização - Controle malha aberta.

**Exemplo 1** Malha aberta

In [ ]:
class OpenLoopExperiment(Simulation):
    def setup(self):
        # Inicializa o robô Pioneer
        self.robot = Pioneer(self.sim, robot_path='/PioneerP3DX')
        print("Open-loop initialized: Moving forward.")

    def loop(self):
        # Definição das velocidades em malha aberta
        # v: velocidade linear em metros por segundo
        # omega: velocidade angular em radianos por segundo
        v = 0.3  
        omega = 1.0  
        
        # Envia o comando constante para os motores
        self.robot.send_commands(v, omega)
        
        # Imprime a telemetria atual para acompanhamento
        current_pose = self.robot.get_pose()
        print(f"Time: {self.current_time:.2f}s | Position: X={current_pose[0]:.2f}m, Y={current_pose[1]:.2f}m")


# ==============================================================================
# EXECUÇÃO
# ==============================================================================
if __name__ == "__main__":
    # Configurado para rodar a simulação por 5 segundos
    experiment = OpenLoopExperiment(max_time=10.0)
    experiment.run()

**Exemplo 2** Controlador em malha fechada

In [15]:
class ClosedLoopExperiment(Simulation):
    def setup(self):
        # Inicializa o robô Pioneer
        self.robot = Pioneer(self.sim, robot_path='/PioneerP3DX')
        self.controller = LyapunovController(k_rho=0.8, 
                                             k_alpha=2.0, 
                                             k_beta=-0.3)

        # Target Dummy
        self.Target=  Dummy(self.sim, obj_path='/target')
        print("Open-loop initialized: Moving forward.")

    def loop(self):
        actual_pose = self.robot.get_pose()
        #Pushing the pose of the target
        SP = self.Target.get_pose()
        # Controlador
        v_cmd, w_cmd = self.controller.calculate_control(actual_pose, SP)
        
        # Envia o comando constante para os motores
        self.robot.send_commands(v_cmd, w_cmd)
        
        # Imprime a telemetria atual para acompanhamento
        current_pose = self.robot.get_pose()
        #print(f"Time: {self.current_time:.2f}s | Position: X={current_pose[0]:.2f}m, Y={current_pose[1]:.2f}m")


# ==============================================================================
# EXECUÇÃO
# ==============================================================================
if __name__ == "__main__":
    simulation_time = 30 #seconds
    
    # Configurado para rodar a simulação por 5 segundos
    experiment = ClosedLoopExperiment(max_time=simulation_time)
    experiment.run()

Open-loop initialized: Moving forward.
CoppeliaSim synchronous simulation initialized.
Simulation execution halted at 30.00 seconds.


# 2. Segunda Etapa: Cadeias de Markov

## 2.1. Cadeias de Markov

Trata-se de um processo estocástico discreto no tempo, denotado por ${\mathbf{S_t}}_{t>0}$, que obedece à rigorosa **Propriedade de Markov**, no qual a informação atual é suficiente para definir o estado futuro, ou seja, o futuro depende apenas do estado presente. Matematicamente:
$$
    \mathbb{P}(S_{t+1} = s_{t+1} | S_t = s_t, S_{t-1}=s_{t-1},\dots, S_0 = s_0) = \mathbb{P}(S_{t+1}=s_{t+1}|S_t = s_t)
$$

Uma cadeia de Markov homogénea no tempo é completamente definida por uma tupla $(S,P)$, onde P é a matriz de transição de probabilidades. Cada elemento $P_{ij}$ representa a probabilidade de transitar do estado $i$ para o estado $j$:

$$
    P_{ij} = \mathbb{P}(S_{t+1}=j \mid S_t = i), \quad \text{onde} \quad \sum_{j\in S} P_{ij} = 1,
    \\
    \forall i \in S
$$


## 2.2. Processos de Decisão de Markov (MDP)

Para modelar o controlo autónomo, estende-se a Cadeia de Markov para permitir a intervenção de um agente, resultando no **Processo de Decisão de Markov (MDP)**. O MDP é formalizado pela tupla $ \mathcal{M} = (\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma) $

Onde:
- $\mathcal{S}$: Espaço de estados do agente, ou seja, as possíveis configurações que ele pode estar;
- $\mathcal{A}$: Espaço das ações que o agente pode realizar;
- $\mathcal{P}$: Matriz de Transição de probabilidades;
- $\mathcal{R}$: Função de recompensa $\mathcal{R}(s,a,s^´)$;
- $\mathcal{\gamma}$: Desconto Temporal, $\mathcal{\gamma} \in [0,1]$.

O objetido do agente é encontrar uma política ótima $\pi^* (a|s)$ que maximize a esperança do Retorno Descontado $G_t$:

$$
    G_t = \sum_{k=0}^{\infty}\gamma^k R_{t+k+1}
$$


## 2.2.1. Função Valor (V) e Função Valor de Ação (Q)

A **Função Valor de Estado** $V^\pi(s)$ define o ganho esperado partindo do estado $s$ sob a política $\pi$:
\begin{equation}
V^\pi(s) = \mathbb{E}_\pi [G_t \mid S_t = s]
\end{equation}
A **Função Valor de Ação** $Q^\pi(s,a)$ avalia a qualidade de tomar a ação $a$ no estado $s$:
\begin{equation}
Q^\pi(s,a) = \mathbb{E}_\pi \left[ \sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t = s, A_t = a \right]
\end{equation}

A **Equação de Otimização de Bellman** estabelece que o valor ótimo de um estado-ação é a recompensa imediata somada ao valor descontado do melhor estado-ação subsequente possível:
\begin{equation}
Q^*(s, a) = \sum_{s' \in \mathcal{S}} \mathcal{P}_{ss'}^a \left( \mathcal{R}_{ss'}^a + \gamma \max_{a'} Q^*(s', a') \right)
\end{equation}

## 2.3. Q-learning

O algoritmo **Q-Learning** contorna o desconhecimento das probabilidades de transição $\mathcal{P}_{ss'}^a$ utilizando amostras reais da interação. A matriz $Q$ é iterativamente atualizada através do Erro de Diferença Temporal (Erro TD, $\delta_t$):
\begin{equation}
\delta_t = \underbrace{R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a')}_{\text{Alvo TD}} - \underbrace{Q(S_t, A_t)}_{\text{Estimativa Atual}}
\end{equation}
\begin{equation}
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \delta_t
\end{equation}

# 3. Terceira Etapa: True Online SARSA ($\lambda$) com Traços de Elegibilidade

# 4. Quarta Etapa: Arquitetura TOSL - iBSR

# 5. Quinta Etapa: Análise estatísticas dos resultados